In [ ]:
!pip install -q transformers accelerate pypdf
!pip install -U bitsandbytes

from transformers import AutoTokenizer, AutoModelForCausalLM
from pypdf import PdfReader
import torch, re, json
from pathlib import Path

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 14.7 MB/s eta 0:00:00


In [ ]:
from google.colab import files
from pathlib import Path
import re, json

# Option A: upload PDF from your machine
print("Upload your PDF…")
up = files.upload()  # pick your file in the dialog
pdf_name = next(iter(up.keys()))
PDF_PATH = Path(pdf_name)

# Option B: if your file is already in the workspace/Drive, set it manually:
# PDF_PATH = Path("/content/your_file.pdf")
print("Using:", PDF_PATH)

In [ ]:
from pypdf import PdfReader

def pdf_to_text(path: Path) -> str:
    reader = PdfReader(str(path))
    return "\n\n".join((p.extract_text() or "") for p in reader.pages)

def clean_text(t: str) -> str:
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)                 # join hyphen line-breaks
    t = re.sub(r"[ \t]*\n(?!\s*\n)", " ", t)               # collapse single newlines
    t = re.sub(r"\s+\n", "\n", t)
    return t.strip()

raw = pdf_to_text(PDF_PATH)
text = clean_text(raw)

# Simple "chapter" split: split on common headings; fallback to big chunks if no headings
parts = re.split(r"(?i)(?:\n\s*(?:Kapitel|Chapter)\s+\d+\b|^\s*\d+(?:\.\d+)*\s+[^\n]+$)", text, flags=re.M)
parts = [p.strip() for p in parts if len(p.split()) > 80]  # keep only substantive parts
print(f"Segments detected: {len(parts)}")


Segments detected: 68


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16
)

tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", quantization_config=bnb
)

def generate(text: str, max_new_tokens=512, temperature=0.7):
    inputs = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature, do_sample=True, top_p=0.9
    )
    return tok.decode(out[0], skip_special_tokens=True)


ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`

In [ ]:
OUT_QA = Path("domain_qa.jsonl")

PRONOUN_BAD = {"dies", "dieses", "diese", "das", "es", "sie", "er", "unser", "unsere", "unserer", "ihr", "ihre", "grundsätzlich", "in deutschland"}
Q_START_OK = ("Was ", "Welche ", "Wie ", "Wer ", "Worin ", "Woraus ", "Weshalb ", "Wozu ")

def is_meaningful_qa(qa: dict) -> bool:
    q = (qa.get("question") or "").strip()
    a = (qa.get("answer") or "").strip()
    if not q.endswith("?"): return False
    if not (12 <= len(q) <= 200): return False
    if not (20 <= len(a) <= 800): return False
    if not q.startswith(Q_START_OK): return False
    # Special case: "Was ist X?" — ensure X is not just a pronoun / deictic
    m = re.match(r"Was\s+ist\s+(.+?)\?\s*$", q, flags=re.I)
    if m:
        x = re.sub(r"[^\wÄÖÜäöüß\- ]", "", m.group(1)).strip().lower()
        # require at least one space or a long token (heuristic for concept-ness)
        if (x in PRONOUN_BAD) or (len(x) < 4 and " " not in x):
            return False
    return True

def extract_json_list(s: str):
    # try to pull a JSON list from the model output
    m = re.search(r"\[\s*\{.*\}\s*\]", s, flags=re.S)
    if not m: return None
    try:
        data = json.loads(m.group(0))
        # normalize keys
        fixed=[]
        for x in data:
            q = x.get("question") or x.get("frage") or x.get("Q") or x.get("Frage")
            a = x.get("answer")   or x.get("antwort") or x.get("A") or x.get("Antwort")
            if q and a:
                fixed.append({"question": str(q).strip(), "answer": str(a).strip()})
        return fixed
    except Exception:
        return None

def make_prompt(section: str, n=3):
    # Keep it VERY explicit to reduce hallucinations and force JSON
    section = section[:2000]
    return (
        "Du bist ein Assistent, der aus dem Abschnitt sinnvolle Prüfungsfragen generiert.\n"
        "Erstelle exakt {n} Frage-Antwort-Paare (Q&A) AUF DEUTSCH.\n"
        "Regeln:\n"
        " - Fragen müssen fachlich und überprüfbar sein (Definitionen, Pflichten, Verfahren, Kennzahlen).\n"
        " - Antworte prägnant (1–3 Sätze) nur aus dem Abschnitt.\n"
        " - GIB AUSSCHLIESSLICH JSON-LISTE zurück: "
        '[{"question":"…?","answer":"…"}, ...]\n\n'
        f"Abschnitt:\n{section}\n"
    ).format(n=n)

all_qas = []
for i, sec in enumerate(parts[:12], 1):   # change slice to cover more segments
    prompt = make_prompt(sec, n=3)
    out = generate(prompt, max_new_tokens=512, temperature=0.4)  # lower temp = more factual
    items = extract_json_list(out) or []
    # filter + dedup
    for qa in items:
        if is_meaningful_qa(qa):
            key = (qa["question"].lower(), qa["answer"].lower())
            if key not in {(x["question"].lower(), x["answer"].lower()) for x in all_qas}:
                all_qas.append(qa)
    print(f"Segment {i}: +{len(items)} (kept {len(all_qas)})")

with OUT_QA.open("w", encoding="utf-8") as f:
    for qa in all_qas:
        f.write(json.dumps(qa, ensure_ascii=False) + "\n")

print(f"✅ Wrote {len(all_qas)} Q&As → {OUT_QA}")
